# RDS Lab 12: Epistemic Parity and SynRD

A lingering question in the prior lab was *what actually makes private synthetic data "good"? What does "good" even mean?*

This is actually a *very* open question in the literature right now.

### Private synthetic data
Much of the private synthetic data research in past decade has been on designing general DP data synthesizers that **model the entire data distribution, inject noise, then sample the noisy model to generate synthetic datasets**

These synthesizers are designed with the intent to be broadly usable in a variety of unanticipated applications - *however*, evidence to support claims of general utility is typically just given as results common public datasets (like Adult or something).

### Evidence of Utility
Limited empirical evidence on relevant tasks undermines trust in the practical use of DP.

Th primary example of public scrutiny and distrust in DP synthetic data is with US Census Bureau, which adopted DP for disclosure avoidance in the 2020 census.

This adoption of DP for the Census was met with resistance among many in the research community, who contend that data infused with DP noise affects demographic totals (Ruggles et al., https://www.aeaweb.org/articles?id=10.1257/pandp.20191107) and exacerbates underrepresentation of minorities (Kenny et al. https://pubmed.ncbi.nlm.nih.gov/34613778/). See also Steed et al. https://www.science.org/stoken/author-tokens/ST-689/full.

The census case has been scrutinized for years, and progress in understanding what went well (and what didn't) has been studied and publicized. But it certainly begs the question: for the rest of the social sciences and medicine, is DP synthetic data viable?

<img src='https://drive.google.com/uc?id=1zNRdiMcl47_2upjjN9aH_8sElRyMel1R'>

### Aligned Benchmarks
To some extent, resistance from these communities is understandable. Current efforts by the DP community (which is primarily theoretical in nature) to evaluate their methodologies in a variety of settings (outside of standard ML benchmark datasets), to think hard about context-driven claims of utility, are limited.

Enter **SynRD** (https://arxiv.org/pdf/2208.12700.pdf). In this paper, the authors attempt to evaluate private synthetic data along a novel criteria: how well does it *preserve* known scientific findings? See also https://papers.ssrn.com/sol3/papers.cfm?abstract_id=5906765.

In other words, SynRD instantiates a reproducibility study over a set of scientific results; it compares functional findings (from real papers) that the researchers got on *real* data with the same set of findings run on *private, synthetic* data. By bootstrapping over the real findings, it has a point of comparison and can answer a question like, "How well does synthesizer X work at reproducing findings from paper Y?" This metric is called **Epistemic Parity**.

### Today's lab
Today, you will get a sense of both the challenges in reproducing scientific findings from papers, and in the ability of a private data synthesizer that we've already worked with (DataSynthesizer) to do so successfully.

1. First, we will discuss the *very* general way we define a finding in social science literature.

2. Then, we will briefly examine a programmatic approach to structuring findings and the publications they come from.

3. Finally, we will put these tools into action. On a real, recent social science publication, you will (1) take the real data, (2) bootstrap a finding (of your choice!) on real data, (3) generate synthetic data, and (4) measure how often that finding is reproduced on the synthetic data, relative to the real.

## 0. Defining a "finding"

(1) **Manually identify natural language claims made by the authors as candidates for findings.**

*Though these claims may appear anywhere in the paper, you can cheat a little because most are found in the results section.*

*Domain expertise provides an advantage BUT it should (almost)always be possible for non-expert readers to identify major claims, and these claims should be stated clearly and without jargon.*

(2) **For each claim, identify the quantitative (or qualitative) evidence that supports the claim, recording the variables involved and statistical methods used.**  

(3) **Then re-implement the analysis to (attempt to) reproduce the salient findings and conclusions in the paper over the original, public dataset**

*While in principle this reproduction step is always possible, in practice, it can be difficult or impossible, and may involve guesswork when the computational details are incomplete.*

(4) **If the reproduction was successful, we generate $n \times B$ synthetic datasets representing $B$ bootstrapped trials with different random seeds from the original data of $n$ samples**

*The additional $B$ draws allow us to account for the randomness in the original data in reproducing the target finding.*  

(5) **We then privatize the bootstrapped data and repeat the process to reproduce the target finding with synthetic DP data**

*Although each synthetic dataset could be scaled to any number of records --- recall that we are sampling a privatized model --- we always use the same number of records as the original data for each bootstrap sample.*

(6) **Finally, we contrast the findings based on original and DP data by measuring the proportion of trials, for each method, where a given finding holds.**

## 1. Epsitemic Parity in Action



1. Skim through the paper [Marital satisfaction as a potential moderator of the association between stress and depression](https://www.sciencedirect.com/science/article/pii/S0165032723001118) by Yuze Shi and Mark A. Whisman (2023).

2. Select one of the following findings to attempt to replicate as part of your lab submission.

* "At both Wave I and Wave II, X% of the sample scored ( >= 9 ) on the 11-item CES-D, which corresponds to a score of ( >= 16 ) on the 20-item version (Kohout et al., 1993), used to define clinically elevated depressive symptoms (Radloff, 1977)."
* "Marital satisfaction and life events were significantly correlated with depressive symptoms at baseline and follow-up."
* "Marital satisfaction and life events were significantly associated with depressive symptoms in the cross-sectional analysis (Table 1). (use Pearson correlation)"

3. Do your best to replicate the finding in the _finding_ function below, and then bootstrap over the real data using the given function.

3. Then, privatize the data, and run the same finding to calculate epistemic parity.

4. Compare and contrast the results


## 2. Programming a Publication

In [7]:
from numpy.random.mtrand import f
import json
import pandas as pd
import numpy as np
from scipy.stats import laplace
from scipy.stats import pearsonr

class Finding():
    def __init__(self, finding_lambda, args=None, description=None, text=None):
        self.finding_lambda = finding_lambda
        self.args = args
        self.description = description
        self.text = text

    def run(self):
        if self.args is None:
            return self.finding_lambda()
        return self.finding_lambda(**self.args)

    def __str__(self):
        return self.description

class Publication():
    DEFAULT_PAPER_ATTRIBUTES = {
        'title': None,
        'id': None,
        'length_pages': 0,
        'authors': [],
        'journal': None,
        'year': 0,
        'current_citations': 0,
    }

    FINDINGS = []

    def __init__(self, dataframe=None, description=None):
        if dataframe is not None:
            self.dataframe = dataframe
            self.real_dataframe = dataframe
        else:
            raise ValueError("Must set dataframe to initialize a paper class.")

        self._description = description
        self.columns = self.real_dataframe.columns

    def run_all_findings(self):
        results = {}
        for finding in self.FINDINGS:
            result = finding.run()
            results[str(finding)] = result
        return results

class Yuze2023Marital(Publication):
    DEFAULT_PAPER_ATTRIBUTES = {
        'title': 'Marital satisfaction as a potential moderator of the association between stress and depression.',
        'id': 'yuze23marital',
        'length_pages': 4,
        'authors': ['Yuze Shi', 'Mark A. Whisman'],
        'journal': 'Journal of Affective Disorders',
        'year': 2023,
        'current_citations': 6,
    }

    def __init__(self, dataframe):
        super(Yuze2023Marital, self).__init__(dataframe=dataframe)

        self.FINDINGS = self.FINDINGS + [
            Finding(self.finding_1_1, description="finding_1_1",
                    text="""(Exact sentence of the finding)""")]

    def _recreate_dataframe(self, filename='yuze2023marital_dataframe.pickle'):
        pass

    def finding_1_1(self):
      df_filtered = self.dataframe.replace(-99, np.nan).dropna()
      corr_mar_w1, p_mar_w1 = pearsonr(df_filtered['w1_marital_satisfaction'], df_filtered['w1_depressive_symptoms'])
      corr_life_w1, p_life_w1 = pearsonr(df_filtered['between_w1w2_life_events'], df_filtered['w1_depressive_symptoms'])
      print(f'Correlation, marital satisfaction and depressive symptoms at w1: {corr_mar_w1}({p_mar_w1})')
      print(f'Correlation, life events and depressive symptoms at w1: {corr_life_w1}({p_life_w1})')

      corr_mar_w2, p_mar_w2 = pearsonr(df_filtered['w2_marital_satisfaction'], df_filtered['w2_depressive_symptoms'])
      corr_life_w2, p_life_w2 = pearsonr(df_filtered['between_w1w2_life_events'], df_filtered['w2_depressive_symptoms'])
      print(f'Correlation, marital satisfaction and depressive symptoms at w2: {corr_mar_w2}({p_mar_w2})')
      print(f'Correlation, life events and depressive symptoms at w2: {corr_life_w2}({p_life_w2})')

      corr_mar_w4, p_mar_w4 = pearsonr(df_filtered['w4_marital_satisfaction'], df_filtered['w4_depressive_symptoms'])
      corr_mar_w5, p_mar_w5 = pearsonr(df_filtered['w5_marital_satisfaction'], df_filtered['w5_depressive_symptoms'])
      print(f'Correlation, marital satisfaction and depressive symptoms at w4: {corr_mar_w4}({p_mar_w4})')
      print(f'Correlation, marital satisfaction and depressive symptoms at w5: {corr_mar_w5}({p_mar_w5})')

      significant_base = (p_mar_w1 < 0.05 and p_life_w1 < 0.05) or (p_mar_w2 < 0.05 and p_life_w2 < 0.05)
      significant_follow = (p_mar_w4 < 0.05) or (p_mar_w5 < 0.05)
      return 1 if significant_base and significant_follow else 0


### Load Data from Paper (ACL survey, already pre-processed)

In [8]:
url = 'https://dataverse.harvard.edu/api/access/datafile/10100393'

from urllib.request import Request, urlopen
req = Request(url)
req.add_header('User-Agent', 'Mozilla/5.0 (X11; Ubuntu; Linux x86_64; rv:77.0) Gecko/20100101 Firefox/77.0')
content = urlopen(req)

df = pd.read_csv(content, sep='\t')
df_without_nulls = df.dropna()
df_without_nulls

,sex,w1_marital_satisfaction,w1_loved_cared,w1_listen_talk,w1_depressive_symptoms,w2_marital_satisfaction,w2_loved_cared,w2_listen_talk,w2_depressive_symptoms,between_w1w2_life_events,...,w3_depressive_symptoms,w4_marital_satisfaction,w4_loved_cared,w4_listen_talk,w4_depressive_symptoms,w5_marital_satisfaction,w5_loved_cared,w5_listen_talk,w5_depressive_symptoms,cesd11
0,1,1,1,2,0.3595,1,1,2,-1.112300,0,...,-0.8938,2,1,2,-0.854986,1,1,2,-1.112278,-2.0
1,1,1,2,2,0.1397,2,2,2,0.978100,0,...,-0.0743,2,2,1,0.590867,2,3,1,0.236583,5.0
2,1,1,1,1,-1.1123,1,1,2,1.055200,0,...,-1.1123,1,1,1,-0.893755,-99,-99,-99,-0.893755,6.0
3,1,1,1,1,0.8105,2,1,1,1.291699,1,...,1.2917,-99,-99,-99,-99.000000,-99,-99,-99,-99.000000,7.0
4,2,1,1,1,-1.1123,1,1,2,-1.112300,0,...,-1.1123,1,1,1,-1.112278,1,1,1,-0.896731,-2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1464,2,2,2,2,-0.8496,2,2,1,-0.634100,1,...,-0.8967,2,2,2,-0.164501,-99,-99,-99,-99.000000,0.0
1465,2,3,4,2,-1.1123,2,2,1,-0.634100,1,...,-0.8967,-99,-99,-99,-99.000000,-99,-99,-99,-99.000000,0.0
1466,2,1,1,2,0.9295,1,2,2,0.763400,0,...,2.5532,-99,-99,-99,0.422056,-99,-99,-99,-99.000000,5.0
1467,2,2,2,2,-0.5564,2,2,3,-0.591500,0,...,-0.5564,2,1,1,-1.112278,3,3,3,-1.112278,-1.0


In [ ]:
## if the code above doesn't work (it thinks you're a bot):
## download the data from the dataverse url https://dataverse.harvard.edu/api/access/datafile/10100393
## upload the .tab file to your working space and run the code below:

#df = pd.read_csv('yuze2023marital.tab', sep='\t')
#df_without_nulls = df.dropna()
#df_without_nulls

In [9]:
# Reproducing the original finding
# You should get 1 (replicated) from your implemented findings function
yuze2023marital = Yuze2023Marital(dataframe=df_without_nulls)
yuze2023marital.run_all_findings()

Correlation, marital satisfaction and depressive symptoms at w1: 0.26889378985439344(2.346965017165055e-08)
Correlation, life events and depressive symptoms at w1: 0.20913622128763892(1.6269637019835918e-05)
Correlation, marital satisfaction and depressive symptoms at w2: 0.3170911745631362(3.233290738751212e-11)
Correlation, life events and depressive symptoms at w2: 0.1562652765613828(0.0013506497617092436)
Correlation, marital satisfaction and depressive symptoms at w4: 0.28063140630734024(5.2810458817761325e-09)
Correlation, marital satisfaction and depressive symptoms at w5: 0.2324936937038475(1.5466540564369036e-06)


{'finding_1_1': 1}

## **3. A bayesian bootstrap (one implementation)**

This part accounts for randomness in the original data. [Bootstrapping](https://en.wikipedia.org/wiki/Bootstrapping_(statistics)) generates new datasets from the same (empirical) data distribution by resampling from the observed data, allowing us to assess whether the findings hold across other plausible realizations of the data.

The following function implements a Bayesian bootstrap procedure to generate B such datasets.

In [10]:
# A bayesian boostrap uses an equally weighted random dirichlet prior in resampling.
# What does that actually mean? Create a vector of length n, where each entry is from a dirichlet random vector
# (guaranteed to add up to 1), and the dirichlet alpha is symmetric and all 1s
def bayesian_bootstrap(data, num_iterations):
    num_rows = len(data)

    bootstrap_samples = []

    for _ in range(num_iterations):
        weights = np.random.dirichlet(np.ones(num_rows), size=1).flatten()
        weights /= weights.sum()
        sample_indices = np.random.choice(range(num_rows), size=num_rows, replace=True, p=weights)
        bootstrap_sample = data.iloc[sample_indices]
        bootstrap_samples.append(bootstrap_sample)

    result = pd.concat(bootstrap_samples, axis=0).reset_index(drop=True)

    return result

# TODO: Bootstrap over the *real* data

In [11]:
# NOTE: this is the ``bootstrap'' param for the lab - no need to change
B = 100

In [13]:
# TODO: ``bootstrap'' over the original data to evaluate your findings ``natural'' replicability
# TODO: Generate the boostrapped data using the bayesian_bootstrap function
bootstrapped_real_df = bayesian_bootstrap(df_without_nulls, B)

In [14]:
bootstrapped_real_df.shape, df_without_nulls.shape

((143900, 23), (1439, 23))

# TODO: Run finding on bootstrapped data samples

In [15]:
# TODO: fix this loop to do sampling from the bootstrapped real data
# then initialize your publication class and it over the bootstrapped sample
# each sample should be of size n = len(real_data)
samples = [bootstrapped_real_df.sample(len(df_without_nulls)) for _ in range(B)]
bootstrapped_finding = []

for sample in samples:
  yuze2023marital = Yuze2023Marital(dataframe=sample)
  result_dict = yuze2023marital.run_all_findings()
  bootstrapped_finding.append(result_dict['finding_1_1'])



Correlation, marital satisfaction and depressive symptoms at w1: 0.24717358840579579(2.539155372823019e-07)
Correlation, life events and depressive symptoms at w1: 0.24571291136532034(3.0014534488676736e-07)
Correlation, marital satisfaction and depressive symptoms at w2: 0.4051002873351219(3.550359549778321e-18)
Correlation, life events and depressive symptoms at w2: 0.20316718834349734(2.49601388568948e-05)
Correlation, marital satisfaction and depressive symptoms at w4: 0.2717719118612929(1.2929540402646326e-08)
Correlation, marital satisfaction and depressive symptoms at w5: 0.24405774216013823(3.623214881038926e-07)
Correlation, marital satisfaction and depressive symptoms at w1: 0.25288238071685304(9.614185598497855e-08)
Correlation, life events and depressive symptoms at w1: 0.22664964311373184(1.8905313067696188e-06)
Correlation, marital satisfaction and depressive symptoms at w2: 0.32605023915678083(3.499555097301079e-12)
Correlation, life events and depressive symptoms at w2:

In [16]:
#TODO: take the average with the result dict to aggregate the reproducibility i.e.
# epsitemic parity for the finding you have implemented (on real data)
prop = np.mean(bootstrapped_finding)
print(f'Proportion of reproduced findings over the bootsrapped real data: {prop}')

Proportion of reproduced findings over the bootsrapped real data: 0.96


## 4. Privatize data

### Setup DataSynthesizer, as before (this is exact code from previous labs...)

In [17]:
!pip install DataSynthesizer
!pip install smartnoise-synth

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.3/114.3 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 48.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.5/144.5 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.6/39.6 MB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.0/101.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 kB 3.5 MB/s eta 0:00:00
  Attempting uninstall: toolz
    Found existing installation: toolz 0.12.1
    Uninstalling toolz-0.12.1:
      Successfully uninstalled toolz-0.12.1
  Attempting uninstall: absl-py
    Found existing installatio

In [18]:
from DataSynthesizer.DataDescriber import DataDescriber
from DataSynthesizer.DataGenerator import DataGenerator
from DataSynthesizer.ModelInspector import ModelInspector
from DataSynthesizer.lib.utils import read_json_file, display_bayesian_network

In [19]:
import DataSynthesizer.DataGenerator as dg
dg.np = np  # Inject np into DataGenerator's global scope

### Apply DataSynthesizer

You can ignore the following cells, doing the necessary transformation privately

In [20]:
#@markdown
from snsynth.transform import TableTransformer, BinTransformer, LabelTransformer #, NoTransformer

EPSILON_BUCKETING = 1.0
BINS = 10
output_data_size = len(df_without_nulls) * B

columns = df.columns
transformers = []
for col in columns:
    if 'depressive_symptoms' in col:
        bin_transformer = BinTransformer(bins=BINS, lower=np.min(df[col]), upper=np.max(df[col]))
        transformers.append(bin_transformer)
    else:
        transformers.append(LabelTransformer())

tt = TableTransformer(transformers)

df_encoded = tt.fit_transform(df, epsilon=EPSILON_BUCKETING)
df_encoded = pd.DataFrame(df_encoded, columns=df.columns)

sens_data_file = 'yuze2023marital.csv'
df_encoded.to_csv(sens_data_file, sep=',')

print('Total epsilon:', EPSILON_BUCKETING)


Total epsilon: 1.0


In [21]:
describer = DataDescriber()
generator = DataGenerator()
description_files = {'correlated attribute mode':     'description(correlated).json'}
synthetic_data_files = {'correlated attribute mode':  'synthetic data(correlated).csv'}

In [22]:
# This might tak
describer.describe_dataset_in_correlated_attribute_mode(sens_data_file,
                                                        epsilon=1.0,
                                                        k=2)
describer.save_dataset_description_to_file(description_files['correlated attribute mode'])
generator.generate_dataset_in_correlated_attribute_mode(n=output_data_size,
                                                        description_file=description_files['correlated attribute mode'],
                                                        seed=0)
generator.save_synthetic_data(synthetic_data_files['correlated attribute mode'])
synthetic_correlated = pd.read_csv(synthetic_data_files['correlated attribute mode'])

================ Constructing Bayesian Network (BN) ================
Adding ROOT w3_listen_talk
Adding attribute w3_loved_cared
Adding attribute w4_listen_talk
Adding attribute w4_marital_satisfaction
Adding attribute w3_depressive_symptoms
Adding attribute w2_depressive_symptoms
Adding attribute w4_depressive_symptoms
Adding attribute between_w1w2_life_events
Adding attribute w5_depressive_symptoms
Adding attribute cesd11
Adding attribute w2_marital_satisfaction
Adding attribute w5_marital_satisfaction
Adding attribute w2_listen_talk
Adding attribute w3_marital_satisfaction
Adding attribute w5_listen_talk
Adding attribute sex
Adding attribute w1_marital_satisfaction
Adding attribute w1_loved_cared
Adding attribute w5_loved_cared
Adding attribute w4_loved_cared
Adding attribute w2_loved_cared
Adding attribute w1_depressive_symptoms
Adding attribute w1_listen_talk
========================== BN constructed ==========================


In [23]:
rng = np.random.default_rng(seed=0)

def custom_unbucket(data, transformers):
    unbucketed_data = data.copy()
    for index, row in data.iterrows():
        for col, transformer in zip(data.columns, transformers):
            if not isinstance(transformer, BinTransformer):
                continue
            bin_width = (transformer.upper - transformer.lower) / transformer.bins
            unbucketed_data.at[index, col] = rng.uniform(transformer.lower + row[col] * bin_width,
                                                        transformer.lower + (row[col] + 1) * bin_width)
    return unbucketed_data

synthetic_correlated.drop(['Unnamed: 0'],axis=1, inplace=True)
df_synthetic = custom_unbucket(pd.DataFrame(synthetic_correlated.values, columns=df.columns), tt.transformers)

#### Here's the synthetic data to compare against
`df_synthetic` is already oversampled to be $n*BOOSTRAP$ length, so you only need to sample proportionally to $n$

In [24]:
df_synthetic

,sex,w1_marital_satisfaction,w1_loved_cared,w1_listen_talk,w1_depressive_symptoms,w2_marital_satisfaction,w2_loved_cared,w2_listen_talk,w2_depressive_symptoms,between_w1w2_life_events,...,w3_depressive_symptoms,w4_marital_satisfaction,w4_loved_cared,w4_listen_talk,w4_depressive_symptoms,w5_marital_satisfaction,w5_loved_cared,w5_listen_talk,w5_depressive_symptoms,cesd11
0,0.0,3.0,4.0,2.0,4.264357,0.0,0.0,1.0,0.599467,2.0,...,-5.753026,3.0,3.0,3.0,-98.829635,5.0,3.0,6.0,-90.563370,4.0
1,1.0,0.0,1.0,0.0,0.462897,1.0,3.0,0.0,1.309093,3.0,...,1.348276,3.0,2.0,3.0,-0.625524,4.0,4.0,6.0,-89.299829,1.0
2,1.0,1.0,1.0,0.0,-0.722565,3.0,1.0,1.0,-1.134746,1.0,...,-90.156887,6.0,1.0,5.0,-5.882945,4.0,1.0,5.0,-91.430765,16.0
3,0.0,0.0,4.0,1.0,0.611524,3.0,3.0,3.0,-0.147188,0.0,...,-93.415466,1.0,3.0,1.0,-3.139749,4.0,5.0,2.0,-1.251768,14.0
4,1.0,1.0,1.0,1.0,1.658909,4.0,2.0,2.0,2.645510,1.0,...,0.741080,2.0,4.0,3.0,-92.328853,5.0,0.0,2.0,-92.616173,8.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
143895,1.0,2.0,4.0,3.0,0.023089,1.0,1.0,2.0,2.976653,0.0,...,2.269378,2.0,1.0,1.0,0.143659,0.0,2.0,0.0,-3.043183,4.0
143896,1.0,0.0,4.0,2.0,1.055968,3.0,0.0,1.0,-0.991544,1.0,...,1.901787,6.0,1.0,5.0,-3.911133,3.0,5.0,1.0,-94.203252,8.0
143897,0.0,0.0,3.0,4.0,2.920937,3.0,0.0,3.0,1.823481,1.0,...,-4.526002,6.0,4.0,1.0,-2.054534,4.0,5.0,6.0,-89.073970,1.0
143898,1.0,1.0,4.0,0.0,-0.874153,4.0,0.0,3.0,0.382659,4.0,...,0.983158,6.0,4.0,4.0,-93.568545,4.0,3.0,0.0,-98.366416,20.0


### Bootstrap over synthetic data

In [26]:
# TODO: we've already drawn BOOTSTRAP * n samples,
# so compare findings ...
synthetic_samples = [df_synthetic.sample(len(df_without_nulls)) for _ in range(B)]
synthetic_replicate = []

for sample in synthetic_samples:
  yuze2023marital = Yuze2023Marital(dataframe=sample)
  result_dict = yuze2023marital.run_all_findings()
  synthetic_replicate.append(result_dict['finding_1_1'])
# NOTE: this will look a lot like the above code!

Correlation, marital satisfaction and depressive symptoms at w1: 0.1022391354782629(0.00010225863460156724)
Correlation, life events and depressive symptoms at w1: -0.01614678445037185(0.5405230582462424)
Correlation, marital satisfaction and depressive symptoms at w2: 0.009320643317382334(0.723885185257728)
Correlation, life events and depressive symptoms at w2: -0.03653872189770978(0.16595354950644886)
Correlation, marital satisfaction and depressive symptoms at w4: 0.20940170752079523(1.0093946182799716e-15)
Correlation, marital satisfaction and depressive symptoms at w5: 0.004013359278349618(0.8790988371630015)
Correlation, marital satisfaction and depressive symptoms at w1: 0.06759545769069433(0.010321087011392721)
Correlation, life events and depressive symptoms at w1: -0.036745917497959135(0.16356402133905507)
Correlation, marital satisfaction and depressive symptoms at w2: 0.04002343687408072(0.12912942547267275)
Correlation, life events and depressive symptoms at w2: -0.038230

In [27]:
print(f'Proportion of reproduced findings over the bootsrapped DP data: {np.mean(synthetic_replicate)}')

Proportion of reproduced findings over the bootsrapped DP data: 0.06


### Compare!
**Please write a short description of the differences between the finding(s) on both real and DP data - how many times were they replicated? To what extent? What level of confidence do you have in the private data?**

#*TODO: report results and write short comparison here*

The real data replicated almost all of the findings (96%), while DP data replicated only a few of the findings (6%). I don't think this private data generated is very useful in further investigation because the replication rate is so low.

## TODO: Discuss the tradeoff between epistemic parity and privacy budget

There is a clear tradeoff between epistemic parity and the privacy budget.

When we lower the privacy budget (adding more noise and making the data more private), this makes statistical relationships less stable, and some findings may no longer be significant. As a result, the probability of reproducing the finding decreases. The epistemic privacy becomes lower.